# Fine-tune Nova's voice (SFT-1) on a Kaggle T4

Runs `training/sft.py` with `training/configs/sft1.yaml`: a 16-bit LoRA (r = 16) over Qwen3.5-2B on
`nimbus-bitext-v1`, one epoch, 278 steps. Every 25 steps it measures loss on the validation split and on the two
held-out intents, and generates eight fixed probe prompts greedily, so you can read the replies change. The
adapter, every checkpoint and the run's files go to a private Hub repo named after the settings.

**Before running (Kaggle):** Settings → Accelerator **GPU T4 x2** (one is used), Internet **On**;
Add-ons → Secrets → **`HF_TOKEN`** (write access) ticked, and **`WANDB_API_KEY`** if you track with
Weights & Biases. Check `EXTRA` in cell 1, then **Run All**. Expect about 15 minutes to install and start,
then the training itself; the first run's time goes into DESIGN.md.

**After:** Output → download **`sft-run.zip`** and unzip it into the repo's `training/runs/` folder (it holds
`run_info.json`, `log_history.json` and `probes.jsonl`, no weights).

In [ ]:
# 1. Settings and secrets.
EXTRA = []            # config overrides, e.g. ["r=8", "lora_alpha=8", "train_rows=1000", "eval_steps=10"]
PUSH = True           # checkpoints and the final adapter to a private Hub repo

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # Unsloth trains on one GPU
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")   # the dataset is private, and the adapter is pushed
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    REPORT_TO = "wandb"
except Exception:
    REPORT_TO = "none"   # the run still saves its curves and probes to files; cell 5 plots them
print("tracking:", REPORT_TO)

In [ ]:
# 2. Clone the repo and install the pinned training stack (the versions that passed the GPU risk tests).
import subprocess, sys
REPO_DIR = os.path.abspath("customer-support-SLM")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jagjotsingh244-source/customer-support-SLM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
%pip install -q -r requirements-gpu.txt
if REPORT_TO == "wandb":
    %pip install -q wandb
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Data and mask checks first (a minute, no model): every row fits, and only Nova's reply is trained.
SET = ("--set " + " ".join(EXTRA)) if EXTRA else ""
!python training/sft.py --config training/configs/sft1.yaml --dry-run {SET}

In [ ]:
# 4. Train. Probe replies print at step 0 and every eval_steps; read them as they come.
FLAGS = f"--report-to {REPORT_TO}" + (" --push" if PUSH else "")
!python training/sft.py --config training/configs/sft1.yaml {FLAGS} {SET}

In [ ]:
# 5. The curves: training loss, validation loss and held-out-intent loss on one chart.
import json, pandas as pd, matplotlib.pyplot as plt
from training.sft import load_config, run_name
cfg = load_config("training/configs/sft1.yaml", EXTRA)
NAME = run_name(cfg)
OUT = os.path.join(cfg["output_root"], NAME)
hist = json.load(open(f"{OUT}/log_history.json"))
train = [(h["step"], h["loss"]) for h in hist if "loss" in h]
val = [(h["step"], h["eval_validation_loss"]) for h in hist if "eval_validation_loss" in h]
unseen = [(h["step"], h["eval_unseen_loss"]) for h in hist if "eval_unseen_loss" in h]
fig, ax = plt.subplots(figsize=(8, 4))
for pts, label in [(train, "train"), (val, "validation"), (unseen, "held-out intents")]:
    ax.plot(*zip(*pts), marker="o" if label != "train" else None, label=label)
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title(NAME); ax.legend(); ax.grid(alpha=.3)
plt.show()
print(pd.DataFrame({"validation": dict(val), "held-out intents": dict(unseen)}).round(4).to_string())

In [ ]:
# 6. The probes, step by step. Look for the step where replies read best, and for the moment they start to
#    sound alike: "same opening" counts probes whose first five words match another probe's at that step.
probes = pd.read_json(f"{OUT}/probes.jsonl", lines=True)
for prompt, g in probes.groupby("prompt", sort=False):
    print(f"\n=== {prompt!r}")
    for _, r in g.iterrows():
        print(f"  [{r.step:>3}] ({r.words}w{'' if r.stopped else ', cut off'}) {r.reply}")
probes["opening"] = probes.reply.str.lower().str.split().str[:5].str.join(" ")
summary = probes.groupby("step").agg(mean_words=("words", "mean"), cut_off=("stopped", lambda s: int((~s).sum())),
                                     same_opening=("opening", lambda s: int(s.duplicated(keep=False).sum())))
print("\n", summary.round(1).to_string())

In [ ]:
# 7. What the run was: memory against the DESIGN.md estimate, the parameter count against the hand count, time.
info = json.load(open(f"{OUT}/run_info.json"))
for k in ["run_name", "gpu", "precision", "trainable_params", "hand_count", "wrapped_layers", "memory_gb",
          "train_seconds", "train_loss", "git", "versions"]:
    print(f"{k:>16}: {info[k]}")
print(f"{'estimate':>16}: 4.9 GB (text only) to 5.6 GB (with the vision tower), DESIGN.md")

In [ ]:
# 8. Optional: if an earlier checkpoint reads better than the final step, make it the adapter at the repo root.
BEST_STEP = None      # e.g. 150; checkpoints exist every save_steps
if BEST_STEP is not None:
    from training.sft import publish_checkpoint
    publish_checkpoint(f"{OUT}/checkpoint-{BEST_STEP}", f"{cfg['hub_user']}/{NAME}",
                       f"{NAME}: adapter from step {BEST_STEP}, chosen from the probes")
    print("root adapter is now step", BEST_STEP)

In [ ]:
# 9. The run's small files for the repo (no weights): download sft-run.zip from Output.
import shutil, tempfile
tmp = tempfile.mkdtemp()
shutil.copytree(OUT, f"{tmp}/{NAME}", ignore=shutil.ignore_patterns("checkpoint-*", "*.safetensors", "*.bin",
                                                                     "tokenizer*", "*.jinja", "*config.json"))
shutil.make_archive("/kaggle/working/sft-run", "zip", tmp)
print(sorted(os.listdir(f"{tmp}/{NAME}")))